In [42]:
import pandas as pd
import torch
import transformers
from torch.utils.data import Dataset
import os
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

In [46]:
folder = '/blue/egn6933/apatil2/embeddings/'

files = sorted([f for f in os.listdir(folder) if f.endswith('.pt')])

train_files, test_files = train_test_split(
    files,
    test_size=0.2,
    random_state=42
)

print(len(train_files), len(test_files))

2444 611


In [47]:
# print(type(train), len(train))

In [59]:
# define the dataset class
class EmbeddingDataset(Dataset):
    def __init__(self, folder, files):
        self.folder = folder
        self.files = files
    
    
    def __len__(self):
        
        return len(self.files)
    
    def __getitem__(self, idx):
        batch = torch.load(os.path.join(self.folder, self.files[idx]))
        # print(type(batch), len(batch))
        X = batch['X'][:, 64, :]
        y = batch['Y']
        
        return X, y                        

In [60]:
# define the data loader

train_dataset = EmbeddingDataset(folder, train_files)

test_dataset = EmbeddingDataset(folder, test_files)

train_loader = DataLoader(train_dataset, batch_size = 1, shuffle = True)
test_loader = DataLoader(test_dataset, batch_size = 1, shuffle = False)

In [70]:
import torch.nn as nn

class MLP(nn.Module):
    
    def __init__(self):
        super().__init__()
    
        self.model = nn.Sequential(
            nn.Linear(768, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(512, 256),
            nn.ReLU(),
            
            nn.Linear(256, 1)
        )
    
    def forward(self, x):
        return self.model(x)
    

In [71]:
#training configuration

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = MLP().to(device)

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr = 1e-4
)

epoch = 5

In [72]:
def compute_accuracy(preds, labels):
    preds = torch.sigmoid(preds)
    preds = (preds>0.5).float()
    
    correct = (preds == labels).sum().item()
    total = labels.size(0)
    
    return correct, total

In [73]:
# training loop
# model.train()

for i in range(epoch):
    model.train()
    total_correct = 0
    total_samples = 0
    total_loss = 0
    
    for X,y in train_loader:
        # print(X.shape, y.shape)
        X = X.squeeze().to(device)
        y = y.squeeze().float().to(device)
        
        optimizer.zero_grad()
        
        output = model(X).squeeze()
        # print(output.shape)
        # print(X.shape)
        
        loss = criterion(output, y)
        loss.backward()
        optimizer.step()
        
        total_loss = loss.item()
        
        correct, total = compute_accuracy(output, y)
        total_correct += correct
        total_samples += total

        # break
    epoch_acc = total_correct / total_samples

    print(f"Epoch {i+1} | Loss: {total_loss:.4f} | Accuracy: {epoch_acc:.4f}")

    torch.save(
        model.state_dict(),
        f"/blue/egn6933/apatil2/model_checkpoints/model_epoch_{i+1}.pt"
    )
    

Epoch 1 | Loss: 0.1454 | Accuracy: 0.9359
Epoch 2 | Loss: 0.1997 | Accuracy: 0.9368
Epoch 3 | Loss: 0.1655 | Accuracy: 0.9380
Epoch 4 | Loss: 0.1566 | Accuracy: 0.9389
Epoch 5 | Loss: 0.1642 | Accuracy: 0.9395
